In [1]:
import os
import glob
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF


In [2]:
from google.colab import drive
drive.mount('/content/drive')

#경로설정
train_rgb_dir = "/content/drive/MyDrive/Colab Notebooks/2D Segmentation/Training/RGB"
train_label_dir = "/content/drive/MyDrive/Colab Notebooks/2D Segmentation/Training/Label"

val_rgb_dir = "/content/drive/MyDrive/Colab Notebooks/2D Segmentation/Validation/RGB"
val_label_dir = "/content/drive/MyDrive/Colab Notebooks/2D Segmentation/Validation/Label"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

# -------------------------------
# EDA 함수 정의-데이터 구조·특성·문제점 등을 사전 분석하는 과정
# -------------------------------

# 2. 라벨 읽고 2d numpy배열 변환 RGB일 경우 첫채널만 사용
def load_label(path):
    raw = np.array(Image.open(path))
    if raw.ndim == 3:
        raw = raw[:, :, 0]
    return raw


def run_full_eda(rgb_dir, label_dir):
    rgb_paths = sorted(glob.glob(os.path.join(rgb_dir, "*.png")))
    label_paths = sorted(glob.glob(os.path.join(label_dir, "*.png")))

    # --- 1: 파일 수 체크 ---
    print("RGB images:", len(rgb_paths))
    print("Label images:", len(label_paths))

    # --- 2: 해상도 분석 ---
    sizes = []
    for p in rgb_paths:
        img = Image.open(p)
        sizes.append(img.size)
    print("\nUnique image sizes:", set(sizes))

    # --- 3: 클래스 비율 분석 ---
    class_counts = {i: 0 for i in range(13)}  # 클래스 ID 0~12

    for lp in tqdm(label_paths, desc="Class count"):
        mask = load_label(lp)
        unique, cnt = np.unique(mask, return_counts=True)
        for u, c in zip(unique, cnt):
            class_counts[u] += c
        # --- 클래스 비율 dict로 반환 ---

    total_pixels = sum(class_counts.values())
    if total_pixels == 0:
        raise ValueError("❌ 라벨 이미지에서 픽셀을 읽지 못했습니다.")

    class_pixel_ratio = {k: v / total_pixels for k, v in class_counts.items()}
    return class_pixel_ratio

    total_pixels = sum(class_counts.values())
    if total_pixels == 0:
        raise ValueError("❌ 라벨 이미지에서 픽셀을 읽지 못했습니다. 경로 확인 필요.")

    class_ratio = {k: v / total_pixels * 100 for k, v in class_counts.items()}

    print("\nClass pixel ratio (%):")
    for k, v in class_ratio.items():
        print(f"Class {k}: {v:.3f}%")

    # 클래스 가중치 계산 예시
    ratios = np.array([class_ratio[k] for k in sorted(class_ratio.keys())]) / 100.0
    class_weights = 1.0 / np.log(1.02 + ratios)
    print("\nSuggested class weights:", class_weights)

    # 클래스 분포 그래프
    plt.figure(figsize=(8,4))
    plt.bar(class_ratio.keys(), class_ratio.values())
    plt.title("Class Distribution (%)")
    plt.xlabel("Class ID")
    plt.ylabel("Percent")
    plt.show()

    # 이미지별 클래스 다양성
    diversity = []
    for lp in label_paths:
        mask = load_label(lp)
        diversity.append(len(np.unique(mask)))

    print("각 라벨당 클래스 다양성:", np.mean(diversity))

    # 라벨 값 범위 검사
    all_values = set()
    for lp in label_paths:
        mask = load_label(lp)
        all_values.update(np.unique(mask))

    print("\nAll label values found in dataset:", sorted(list(all_values)))

    return class_weights


class_weights_np = run_full_eda(train_rgb_dir, train_label_dir)

RGB images: 1000
Label images: 1000

Unique image sizes: {(800, 600)}


Class count: 100%|██████████| 1000/1000 [00:18<00:00, 52.95it/s]


In [4]:
# -------------------------------
#segmentation dataset + 클래스 픽셀비율 기반
# -------------------------------

from torch.utils.data import Dataset
import torchvision.transforms as T

# -------------------------------
# 클래스 픽셀 비율 기반 가중치 계산
# -------------------------------
# run_full_eda 결과를 바로 dict으로 변환
class_pixel_array = run_full_eda(train_rgb_dir, train_label_dir)  # array-like
class_pixel_ratio = {i: v for i, v in enumerate(class_pixel_array)}  # dict로 변환

# 픽셀이 적을수록 증강 확률 증가
class_weights = {k: min(max(0.1, 0.5 / (v + 1e-6)), 0.8) for k, v in class_pixel_ratio.items()}
# -------------------------------
# Dataset 정의a
# -------------------------------
class SegDataset(Dataset):
    def __init__(self, rgb_dir, label_dir, base_transform=None):
        self.rgb_paths = sorted([
            os.path.join(rgb_dir, f) for f in os.listdir(rgb_dir) if f.endswith(".png")
        ])
        self.label_paths = sorted([
            os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.endswith(".png")
        ])
        assert len(self.rgb_paths) == len(self.label_paths), "이미지/라벨 개수 불일치!"
        self.base_transform = base_transform

    def __len__(self):
        return len(self.rgb_paths)

    def __getitem__(self, idx):
        # 1. 이미지/마스크 로드
        img = Image.open(self.rgb_paths[idx]).convert("RGB")
        mask = np.array(Image.open(self.label_paths[idx])).astype(np.int64)

        # 2. 이미지에 포함된 클래스 확인 후 augmentation 확률 계산
        unique_classes = np.unique(mask)
        aug_prob = max([class_weights[c] for c in unique_classes])

        # 3. 랜덤 증강 적용
        if random.random() < aug_prob: # (1) 증강 전체가 적용될 확률
            # Horizontal / Vertical Flip
            if random.random() < 0.5: img = T.functional.hflip(img)    # (2) 50% 확률로 수평 뒤집기
            if random.random() < 0.5: img = T.functional.vflip(img)   # (3) 50% 확률로 수직 뒤집기
            # Random rotation
            angle = random.choice([0, 90, 180, 270])
            if angle != 0: img = T.functional.rotate(img, angle)

        # 4. 기본 transform 적용 (Resize + ToTensor + Normalize)
        if self.base_transform:
            img = self.base_transform(img)

        mask = torch.tensor(mask).long()
        return img, mask

# -------------------------------
# 기본 transform 정의
# -------------------------------
train_base_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_base_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])



RGB images: 1000
Label images: 1000

Unique image sizes: {(800, 600)}


Class count: 100%|██████████| 1000/1000 [00:18<00:00, 54.33it/s]


In [5]:
# -------------------------------
# resnet FNC50
# -------------------------------
NUM_CLASSES = 13 #13개의 카테고리

# ResNet Bottleneck 블록만 별도 정의 1×1 conv → 3×3 conv → 1×1 conv 구조
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, mid_channels,
                 stride=1, dilation=1, downsample=None):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, mid_channels,  #차원 축소하기
                               kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)

        self.conv2 = nn.Conv2d(  #3x3 실제 특징 추출
            mid_channels,
            mid_channels,
            kernel_size=3,
            stride=stride,
            padding=dilation,
            dilation=dilation,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(mid_channels)

        self.conv3 = nn.Conv2d(mid_channels,  #차원 복원하기
                               mid_channels * self.expansion,
                               kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x #skip connection 입력 x와 output을 더함

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity  #블록의 입력을 최종출력과 더해주기
        out = self.relu(out)
        return out


class MyFcnResNet50(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        # --------- conv1 부분 ---------  conv1 + maxpool 초기 feature 추출하고 + 공간 크기 줄이기
        self.conv1 = nn.Conv2d(3, 64,
                               kernel_size=7,
                               stride=2,
                               padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3,
                                    stride=2,
                                    padding=1)

        # --------- ResNet layer1~4 (FCN_ResNet50 스타일) --------- bottleneck 블록 반복
        # layer1: in=64, mid=64, blocks=3, stride=1, dilation=1
        self.layer1 = self._make_layer(
            in_channels=64,
            mid_channels=64,
            blocks=3,
            stride=1,
            dilations=[1, 1, 1]
        )

        # layer2: in=256, mid=128, blocks=4, 첫 블록 stride=2
        self.layer2 = self._make_layer(
            in_channels=256,
            mid_channels=128,
            blocks=4,
            stride=2,
            dilations=[1, 1, 1, 1]
        )

        # layer3: in=512, mid=256, blocks=6
        #   block0: dilation=1
        #   block1~5: dilation=2
        self.layer3 = self._make_layer(
            in_channels=512,
            mid_channels=256,
            blocks=6,
            stride=1,
            dilations=[1, 2, 2, 2, 2, 2]
        )

        # layer4: in=1024, mid=512, blocks=3
        #   block0: dilation=2
        #   block1,2: dilation=4
        self.layer4 = self._make_layer(
            in_channels=1024,
            mid_channels=512,
            blocks=3,
            stride=1,
            dilations=[2, 4, 4]
        )

        # --------- FCNHead와 동일한 classifier / aux_classifier ---------
        # main classifier: layer4(2048ch) → 512 → num_classes
        self.classifier = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Conv2d(512, num_classes, kernel_size=1)
        )

        # -------aux classifier: layer3(1024ch) → 256 → num_classes
        self.aux_classifier = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

    # layer1~4 만드는 헬퍼 (클래스 안에 포함)
    def _make_layer(self, in_channels, mid_channels,
                    blocks, stride, dilations):
        assert len(dilations) == blocks

        layers = []

        first_dilation = dilations[0]
        out_channels = mid_channels * Bottleneck.expansion

        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=1,
                          stride=stride,
                          bias=False),
                nn.BatchNorm2d(out_channels),
            )

        # 첫 블록
        layers.append(
            Bottleneck(
                in_channels=in_channels,
                mid_channels=mid_channels,
                stride=stride,
                dilation=first_dilation,
                downsample=downsample
            )
        )

        # 나머지 블록들
        in_channels = out_channels
        for d in dilations[1:]:
            layers.append(
                Bottleneck(
                    in_channels=in_channels,
                    mid_channels=mid_channels,
                    stride=1,
                    dilation=d,
                    downsample=None
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]

        # backbone
        x = self.conv1(x)    # H/2
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)  # H/4

        x = self.layer1(x)   # (B, 256, H/4, W/4)
        x = self.layer2(x)   # (B, 512, H/8, W/8)
        x3 = self.layer3(x)  # (B, 1024, H/8, W/8)
        x4 = self.layer4(x3) # (B, 2048, H/8, W/8)

        # main head
        main = self.classifier(x4)  # (B, C, H/8, W/8)
        main = F.interpolate(main, size=(H, W),
                             mode="bilinear", align_corners=False)

        # aux head
        aux = self.aux_classifier(x3)  # (B, C, H/8, W/8)
        aux = F.interpolate(aux, size=(H, W),
                            mode="bilinear", align_corners=False)

        return {"out": main, "aux": aux}

In [6]:
from torch.utils.data import DataLoader
# -------------------------------
# Dataset 생성
# -------------------------------
train_dataset = SegDataset(train_rgb_dir, train_label_dir, base_transform=train_base_transform)
val_dataset = SegDataset(val_rgb_dir, val_label_dir, base_transform=val_base_transform)


# -------------------------------
# DataLoader 생성
# -------------------------------
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4)




/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [7]:
#최적화 함수와 기기설정, lose계산모델
import torch.optim as optim

if torch.cuda.is_available():
    device = "cuda"
    print(" GPU(CUDA)를 사용")
    print(f"   - GPU 이름: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("GPU를 사용할 수 없습니다. CPU로 실행됩니다.")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MyFcnResNet50(num_classes=13).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

GPU를 사용할 수 없습니다. CPU로 실행됩니다.


In [8]:
import time

EPOCHS = 15

print("Start Training...")
for epoch in range(EPOCHS):
    start_time = time.time()  # epoch 시작 시간
    model.train()
    total_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        # ---------------------------
        # masks 차원 확인 & 3D로 변환
        # (B,1,H,W) -> (B,H,W)
        if masks.ndim == 4 and masks.shape[1] == 1:
            masks = masks.squeeze(1)
        # ---------------------------

        optimizer.zero_grad()

        out_dict = model(imgs)
        out = out_dict["out"]         # main 출력만 가져오기

        loss = criterion(out, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_time = time.time() - start_time
    print(f"[Epoch {epoch+1}] Train Loss: {total_loss/len(train_loader):.4f} | Time: {epoch_time:.1f}s")

Start Training...


RuntimeError: only batches of spatial targets supported (3D tensors) but got targets of dimension: 4

In [ ]:
#-------1. 원본사진, 2. label 색 입힌거, 3. 모델 예측 컬러 마스크
# -------------------------------
# 마스크 컬러화 함수
# -------------------------------
def colorize_mask(mask):
    """
    라벨 마스크를 컬러 이미지로 변환
    mask : (H, W) 형태의 라벨
    """
    palette = np.array([
        [0, 0, 0],          # 0 None
        [70, 70, 70],       # 1 Buildings
        [190, 153, 153],    # 2 Fences
        [150, 120, 90],     # 3 Other
        [220, 20, 60],      # 4 Pedestrians
        [153, 153, 153],    # 5 Poles
        [255, 255, 0],      # 6 RoadLines
        [128, 64, 128],     # 7 Roads
        [244, 35, 232],     # 8 Sidewalks
        [107, 142, 35],     # 9 Vegetation
        [0, 0, 142],        # 10 Vehicles
        [102, 102, 156],    # 11 Walls
        [220, 220, 0],      # 12 TrafficSigns
    ], dtype=np.uint8)
    return palette[mask]

# -------------------------------
# 시각화 함수
# -------------------------------
def visualize_segmentation(model, dataset, device, idx=0):
    """
    idx번째 이미지 시각화:
    1) 원본 이미지
    2) 레이블(mask) 컬러화
    3) 모델 예측 컬러화
    """
    model.eval()
    with torch.no_grad():
        # 1. 이미지/라벨 가져오기
        img, mask = dataset[idx]
        img_input = img.unsqueeze(0).to(device)  # 배치 차원 추가

        # 2. 모델 예측
        out_dict = model(img_input)
        pred_logits = out_dict['out']           # (1, C, H, W)
        pred_mask = torch.argmax(pred_logits, dim=1).squeeze(0).cpu().numpy()  # (H, W)

        # 3. 원본 이미지 (Tensor -> numpy, 0~1)
        img_np = img.permute(1, 2, 0).cpu().numpy()

        # 4. 컬러 마스크
        mask_color = colorize_mask(mask.numpy())
        pred_color = colorize_mask(pred_mask)

        # 5. 시각화
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(img_np)
        axes[0].set_title("Original Image")
        axes[0].axis("off")

        axes[1].imshow(mask_color)
        axes[1].set_title("Ground Truth Mask")
        axes[1].axis("off")

        axes[2].imshow(pred_color)
        axes[2].set_title("Predicted Mask")
        axes[2].axis("off")

        plt.show()


In [ ]:
#iou계산
num_classes = 13

# 1. 클래스 이름
CLASS_NAMES = [
    "None (Background)", # 0
    "Buildings",         # 1
    "Fences",            # 2
    "Other",             # 3
    "Pedestrians",       # 4
    "Poles",             # 5
    "RoadLines",         # 6
    "Roads",             # 7
    "Sidewalks",         # 8
    "Vegetation",        # 9
    "Vehicles",          # 10
    "Walls",             # 11
    "TrafficSigns"       # 12
]

def compute_iou(pred, target, num_classes):
    ious = []
    pred = torch.argmax(pred, dim=1)  # (B, C, H, W) -> (B, H, W)
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union == 0:
            ious.append(np.nan)  # 해당 클래스가 배치에 없으면 제외
        else:
            ious.append(intersection / union)
    return ious

# -------------------------------
#ioU = (겹치는 픽셀 수) / (겹치는 픽셀 포함 모든 픽셀 수)
# Validation mIoU 계산
# -------------------------------
def evaluate_miou(model, val_loader, device):
    model.eval()
    all_ious = []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            batch_ious = compute_iou(out, masks, num_classes)
            all_ious.append(batch_ious)

    all_ious = np.array(all_ious)
    mean_ious = np.nanmean(all_ious, axis=0)  # 클래스별 평균
    miou = np.nanmean(mean_ious)              # 전체 클래스 평균

    # 클래스별 IoU 출력
    for i, iou in enumerate(mean_ious):
        print(f"{CLASS_NAMES[i]:15s} : {iou:.4f}")
    print(f"\최종 mIoU: {miou:.4f}")
    return miou, mean_ious
